<a href="https://colab.research.google.com/github/LeonardooAlves/LSE-ST445/blob/main/Week%203/1_st445_week3_lab1_numpy_SOLUTIONS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST445 Week 3 seminar: NumPy and pandas in practice

**Seminar lab · about 95 minutes, with the extension track for confident students**

The lecture explained how NumPy arrays and pandas tables store data, and where they catch people out. This lab is the hands-on half. You will:

1. **predict, then run** five short cells, each a trap from the lecture;
2. **recap matrices**: create arrays of different shapes, access their values with every kind of indexing, and compute with them, element by element and as matrices;
3. work through NumPy problems in three steps: a **worked example** to read, a **completion problem** with gaps, and an **unaided problem** to write yourself. Together they use **universal functions**, **broadcasting** and **indexing**;
4. do the **applied task**: turn the raw JSON you collected from the World Bank API in Week 2 into a properly typed DataFrame;
5. finish with an **error hunt** on broken broadcasting and chained-assignment code;
6. optionally, take the **extension track**: vectorise a small simulation and compare runtimes.

| Part | Time |
|---|---|
| 1. Predict, then run | 10 min |
| 2. Arrays and matrices: create, index, compute | 20 min |
| 3. Worked example, completion problem, unaided problem | 25 min |
| 4. Applied task | 25 min |
| 5. Error hunt | 10 min |
| 6. Extension | if time allows |

Cell marks, as in earlier labs: 🟢 run and read · 🟡 fill the gaps marked `____` · 🔵 write the code yourself · ✍️ write a short answer in the Markdown cell. Test cells print ✅ or ❌ with a hint. **Method cards** explain each new function and the minimum you need to set.

**Before you start:** **File → Save a copy in Drive**, and rename it `<yourname>_week3_lab.ipynb`.

## Set-up

In [110]:
# 🟢 Run this first
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

def check(condition, hint="Not yet. Read the task again."):
    print("✅ Correct" if condition else f"❌ {hint}")

def filled(*values):
    """False (with a reminder) if any value still contains ____."""
    if any("____" in str(v) for v in values):
        print("✋ Replace every ____ first, then run this cell again.")
        return False
    return True

print("NumPy", np.__version__, "| pandas", pd.__version__)

NumPy 2.1.3 | pandas 2.2.3


**Check the pandas version printed above.** The lecture described pandas 3.0. If Colab shows 2.x, everything in this lab still works; a few outputs differ (text columns show as `object` rather than `str`), and the markdown says where.

---
# Part 1. Predict, then run

Five short cells. For each one: **write your prediction** in the Markdown cell first, then run the code, then read the explanation. Getting a prediction wrong is the point: it shows exactly which mental model needs updating.

## 1.1 A view or a copy?

```python
a = np.arange(6)          # [0 1 2 3 4 5]
b = a[1:4]                # a slice
c = a[a > 3]              # a boolean mask
b[0] = 100
c[0] = -1
print(a)
```

✍️ **Your prediction for `print(a)`:**

In [111]:
# 🟢 Run after predicting
a = np.arange(6)
b = a[1:4]
c = a[a > 3]
b[0] = 100
c[0] = -1
print(a)
print("b shares memory with a:", np.shares_memory(a, b), "| c shares memory with a:", np.shares_memory(a, c))

[  0 100   2   3   4   5]
b shares memory with a: True | c shares memory with a: False


**What happened:** `a` is `[0 100 2 3 4 5]`. The slice `b` is a **view** of the same buffer, so changing `b` changed `a`. The boolean mask `c` is a **copy**, so changing `c` did not. Rule from the lecture: basic slices are views; boolean masks and lists of indices make copies; `.copy()` when the original must not change.

## 1.2 Comparing floats

```python
x = 0.1 * 3
print(x == 0.3)
print(np.isclose(x, 0.3))
```

✍️ **Your prediction for each line:**

In [112]:
# 🟢 Run after predicting
x = 0.1 * 3
print(x == 0.3, "| x is really", repr(x))
print(np.isclose(x, 0.3))

False | x is really 0.30000000000000004
True


**What happened:** `False`, then `True`. 0.1 has no exact binary form, so `0.1 * 3` is `0.30000000000000004`. `==` asks for identical bits; `np.isclose` asks whether two values are within a small tolerance, which is the question you almost always mean.

> **Method card: `np.isclose(a, b, rtol=1e-05, atol=1e-08)`** compares element by element and returns True where `|a − b| ≤ atol + rtol × |b|`. Works on whole arrays. `np.allclose` returns a single True or False for "all close".

## 1.3 Integer overflow

```python
counts = np.array([120, 125], dtype=np.int8)
print(counts + 10)
```

✍️ **Your prediction:**

In [113]:
# 🟢 Run after predicting
counts = np.array([120, 125], dtype=np.int8)
print(counts + 10)
print("int8 range:", np.iinfo(np.int8).min, "to", np.iinfo(np.int8).max)

[-126 -121]
int8 range: -128 to 127


**What happened:** `[-126 -121]`. `int8` holds −128 to 127, so 130 and 135 **wrap round** silently, with no error. Python's own integers would have given 130 and 135. Choose a dtype with room for the values you expect, and check limits with `np.iinfo`.

## 1.4 A misaligned index

```python
population = pd.Series([67.4, 83.1], index=["FRA", "DEU"])
area       = pd.Series([0.36, 0.55, 0.24], index=["DEU", "FRA", "GBR"])   # million km²
print(population / area)
```

✍️ **Your prediction (how many rows, and which are missing?):**

In [114]:
# 🟢 Run after predicting
population = pd.Series([67.4, 83.1], index=["FRA", "DEU"])
area = pd.Series([0.36, 0.55, 0.24], index=["DEU", "FRA", "GBR"])
print(population / area)

DEU    230.833333
FRA    122.545455
GBR           NaN
dtype: float64


**What happened:** three rows, sorted by label: `DEU` 230.8 and `FRA` 122.5 (each divided by **its own** area, even though the two Series list them in different orders), and `GBR` NaN, because there is no population for it. pandas aligned on **labels**, not positions. The NaN is an alignment result, not missing data in either source.

## 1.5 What dtype did pandas infer?

```python
scores = pd.Series(["10", "20", "30"])
print(scores.dtype)
print(scores.sum())
```

✍️ **Your prediction for each line:**

In [115]:
# 🟢 Run after predicting
scores = pd.Series(["10", "20", "30"])
print(scores.dtype)
print(scores.sum())
print(pd.to_numeric(scores).sum())

object
102030
60


**What happened:** the dtype is text (`str` in pandas 3.0, `object` in 2.x), and `sum()` **joins** the strings: `"102030"`. Numbers that arrive as text, from a CSV or a JSON field, stay text until you convert them; `pd.to_numeric` gives 60. Always check `df.dtypes` before computing anything.

---
# Part 2. Arrays and matrices: create, index, compute

A recap of matrices with NumPy, in three steps: **create** arrays of different shapes, **access** their values with each kind of indexing, and **compute** with them, element by element and as matrices. Each step starts with examples to run and read, then exercises.

## 2.1 Creating arrays of different shapes

An array's **shape** is a tuple with its size along each axis: `(5,)` is a vector of 5 values, `(3, 4)` a matrix with 3 rows and 4 columns, `(2, 3, 4)` two stacked 3 × 4 matrices. `ndim` is the number of axes and `size` the number of elements.

> **Method card: creating arrays**
>
> | Function | Creates | Example |
> |---|---|---|
> | `np.array(list)` | an array from a list (a list of lists for a matrix) | `np.array([[1, 2], [3, 4]])` |
> | `np.arange(start, stop, step)` | evenly spaced values, `stop` **excluded** | `np.arange(0, 10, 2)` |
> | `np.linspace(start, stop, num)` | `num` evenly spaced values, `stop` **included** | `np.linspace(0, 1, 5)` |
> | `np.zeros(shape)`, `np.ones(shape)`, `np.full(shape, value)` | an array filled with one value | `np.zeros((2, 3))` |
> | `np.eye(n)` | the n × n **identity matrix** | `np.eye(3)` |
> | `arr.reshape(shape)` | the same values in a new shape (sizes must match) | `np.arange(6).reshape(2, 3)` |
> | `rng.integers(low, high, size)` | random integers from a seeded Generator | `rng.integers(0, 10, (2, 2))` |

In [116]:
# 🟢 Create, then look at shape, ndim, size and dtype
v = np.array([4, 8, 15, 16, 23])                  # a vector
M = np.array([[1, 2, 3], [4, 5, 6]])               # a 2 x 3 matrix: a list of rows
T = np.arange(24).reshape(2, 3, 4)                 # 2 matrices of 3 x 4, stacked
I = np.eye(3)                                      # the 3 x 3 identity matrix
col = np.arange(3).reshape(3, 1)                   # a column vector
row = np.arange(3).reshape(1, 3)                   # a row vector

for name, arr in [("v", v), ("M", M), ("T", T), ("I", I), ("col", col), ("row", row)]:
    print(f"{name:>3}: shape {str(arr.shape):<10} ndim {arr.ndim}  size {arr.size:>2}  dtype {arr.dtype}")
print(I)

  v: shape (5,)       ndim 1  size  5  dtype int64
  M: shape (2, 3)     ndim 2  size  6  dtype int64
  T: shape (2, 3, 4)  ndim 3  size 24  dtype int64
  I: shape (3, 3)     ndim 2  size  9  dtype float64
col: shape (3, 1)     ndim 2  size  3  dtype int64
row: shape (1, 3)     ndim 2  size  3  dtype int64
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


**What you should see:** `v` has shape `(5,)` and one axis; `M` has shape `(2, 3)`; `T` has shape `(2, 3, 4)` and three axes; the identity `I` is float, with 1 on the diagonal and 0 elsewhere. Notice the difference between `v`, with shape `(5,)`, and the column vector `col`, with shape `(3, 1)`: a one-dimensional array is **neither** a row nor a column until you give it a second axis. That distinction matters for matrix products in 2.3.

In [117]:
# 🟢 More ways to create, and reshape
print(np.arange(0, 1, 0.25))                     # [0.   0.25 0.5  0.75]: stop excluded
print(np.linspace(0, 1, 5))                      # [0.   0.25 0.5  0.75 1.  ]: stop included
print(np.full((2, 2), 7))
print(np.arange(12).reshape(3, -1).shape)        # -1 means "work it out": (3, 4)
rng = np.random.default_rng(445)
print(rng.integers(0, 10, size=(2, 3)))          # random integers, the same every run
try:
    np.arange(10).reshape(3, 4)
except ValueError as err:
    print("ValueError:", err)                     # 10 values do not fit a 3 x 4 shape

[0.   0.25 0.5  0.75]
[0.   0.25 0.5  0.75 1.  ]
[[7 7]
 [7 7]]
(3, 4)
[[1 5 5]
 [0 1 7]]
ValueError: cannot reshape array of size 10 into shape (3,4)


**What you should see:** `arange` stops **before** 1 while `linspace` includes it; `reshape(3, -1)` lets NumPy work out the missing size; and reshaping 10 values into 12 places fails, because `reshape` never adds or drops values.

🟡 **Complete the code** to create each array.

In [118]:
a1 = np.arange(10)                         # the integers 0 to 9
a2 = np.zeros((3, 4))                      # a 3 x 4 matrix of zeros
a3 = np.eye(4)                             # the 4 x 4 identity matrix
a4 = np.linspace(0, 10, 6)                 # 6 values from 0 to 10, both included
a5 = np.arange(1, 13).reshape(3, 4)        # 1 to 12 as a matrix with 3 rows and 4 columns
a6 = np.arange(1, 13).reshape(2, 2, 3)     # the same 12 values as 2 matrices of 2 x 3
a7 = np.array([5, 6, 7]).reshape(3, 1)     # [5, 6, 7] as a column vector

if filled(a1, a2, a3, a4, a5, a6, a7):
    for name, arr in [("a1", a1), ("a2", a2), ("a3", a3), ("a4", a4), ("a5", a5), ("a6", a6), ("a7", a7)]:
        print(name, arr.shape)

a1 (10,)
a2 (3, 4)
a3 (4, 4)
a4 (6,)
a5 (3, 4)
a6 (2, 2, 3)
a7 (3, 1)


In [119]:
# Tests for 2.1
check(list(a1) == list(range(10)), "np.arange(10) stops before 10.")
check(a2.shape == (3, 4) and not a2.any(), "Pass the shape as a tuple: np.zeros((3, 4)).")
check(np.array_equal(a3, np.eye(4)), "np.eye(4).")
check(np.allclose(a4, [0, 2, 4, 6, 8, 10]), "np.linspace(0, 10, 6) includes both ends.")
check(a5.shape == (3, 4) and a5[2, 3] == 12, "reshape(3, 4): 3 rows, 4 columns.")
check(a6.shape == (2, 2, 3), "reshape(2, 2, 3): 2 matrices, each 2 x 3.")
check(a7.shape == (3, 1) and a7[2, 0] == 7, "A column vector has shape (3, 1).")

✅ Correct
✅ Correct
✅ Correct
✅ Correct
✅ Correct
✅ Correct
✅ Correct


## 2.2 Accessing values: every kind of indexing

For a matrix, `M[i, j]` is the element in **row `i`, column `j`**, both counted from 0. Negative indices count from the end. A slice `start:stop:step` excludes `stop`, as in Python.

> **Method card: indexing a matrix**
>
> | Expression | Selects | Result |
> |---|---|---|
> | `M[i, j]` | one element | a number |
> | `M[i]` or `M[i, :]` | row `i` | shape `(n_cols,)` |
> | `M[:, j]` | column `j` | shape `(n_rows,)`; `M[:, [j]]` keeps it as a column `(n_rows, 1)` |
> | `M[r0:r1, c0:c1]` | a block | a **view** |
> | `M[::2]`, `M[:, ::-1]` | every other row; columns reversed | a view |
> | `M[M > 5]` | elements where a condition holds | a 1-D **copy** |
> | `M[[0, 2]]` | rows 0 and 2 (fancy indexing) | a copy |
> | `M[[0, 2], [1, 3]]` | the **pairs** (0, 1) and (2, 3) | 2 elements |
> | `M[np.ix_([0, 2], [1, 3])]` | the **sub-matrix** of rows 0, 2 and columns 1, 3 | shape `(2, 2)` |
> | `np.diag(M)` | the main diagonal | 1-D |

In [120]:
# 🟢 A 4 x 4 matrix to index
M = np.arange(1, 17).reshape(4, 4)
print(M, "\n")

print(M[0, 0], M[2, 3], M[-1, -1])         # 1 12 16: single elements
print(M[1])                                 # [5 6 7 8]: row 1
print(M[:, 2])                              # [ 3  7 11 15]: column 2, as a 1-D array
print(M[:, [2]].shape)                      # (4, 1): column 2, kept as a column
print(M[1:3, 1:3])                          # the 2 x 2 block in the middle
print(M[::2])                               # rows 0 and 2
print(M[:, ::-1][0])                        # row 0 with the columns reversed: [4 3 2 1]

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]] 

1 12 16
[5 6 7 8]
[ 3  7 11 15]
(4, 1)
[[ 6  7]
 [10 11]]
[[ 1  2  3  4]
 [ 9 10 11 12]]
[4 3 2 1]


**What you should see:** `M` holds 1 to 16 row by row. `M[2, 3]` is 12: row 2, column 3. Row and column slices return one-dimensional arrays unless you keep the axis with a list (`M[:, [2]]`). `M[1:3, 1:3]` is the central block `[[6, 7], [10, 11]]`.

In [121]:
# 🟢 Boolean masks, fancy indexing, pairs versus sub-matrices, and the diagonal
print(M[M % 2 == 0])                        # even numbers, flattened to 1-D
print(M[M > 12])                            # [13 14 15 16]
print(M[[0, 3]])                            # rows 0 and 3
print(M[[0, 3], [1, 2]])                    # the PAIRS (0, 1) and (3, 2): [ 2 15]
print(M[np.ix_([0, 3], [1, 2])])            # the SUB-MATRIX rows {0, 3} x columns {1, 2}
print(np.diag(M), M.trace())                # diagonal [1 6 11 16], and its sum (the trace): 34

[ 2  4  6  8 10 12 14 16]
[13 14 15 16]
[[ 1  2  3  4]
 [13 14 15 16]]
[ 2 15]
[[ 2  3]
 [14 15]]
[ 1  6 11 16] 34


**What you should see:** a common surprise is the difference between `M[[0, 3], [1, 2]]` and `M[np.ix_([0, 3], [1, 2])]`. Two lists inside the brackets are read as **pairs** of coordinates: (0, 1) and (3, 2), giving two numbers. `np.ix_` builds the **sub-matrix** of those rows and columns, a 2 × 2 result.

🔵 **Exercises.** Use the matrix `M` above. Store each answer in the variable named.

In [122]:
corners = M[np.ix_([0, -1], [0, -1])]         # or M[::3, ::3]
second_col = M[:, [1]]
last_row_rev = M[-1, ::-1]
big = M[M > 10]
anti_diag = np.diag(M[:, ::-1])                # reverse the columns, then take the diagonal

In [123]:
# Tests for 2.2
check(corners is not None and np.array_equal(corners, [[1, 4], [13, 16]]), "np.ix_([0, -1], [0, -1]), or step slices M[::3, ::3].")
check(second_col is not None and second_col.shape == (4, 1) and second_col[3, 0] == 14, "M[:, [1]] keeps the column axis.")
check(last_row_rev is not None and list(last_row_rev) == [16, 15, 14, 13], "Index row -1, then reverse with ::-1.")
check(big is not None and list(big) == [11, 12, 13, 14, 15, 16], "A boolean mask: M[M > 10].")
check(anti_diag is not None and list(anti_diag) == [4, 7, 10, 13], "Reverse the columns, then np.diag.")

✅ Correct
✅ Correct
✅ Correct
✅ Correct
✅ Correct


## 2.3 Arithmetic: element by element, and as matrices

NumPy has **two** kinds of multiplication, and mixing them up is the most common matrix mistake in Python:

| Operation | Symbol | Meaning | Shapes |
|---|---|---|---|
| Element-wise | `A * B` | multiply matching elements | the same shape (or broadcastable) |
| Matrix product | `A @ B` | rows of `A` times columns of `B`, summed | `(m, n) @ (n, p)` gives `(m, p)` |

`+`, `-`, `/` and `**` are always element-wise. A number combined with an array applies to every element (broadcasting).

> **Method card: linear algebra in NumPy**
> * `A.T`: the transpose (rows become columns); a view.
> * `A @ B` (or `np.matmul`): the matrix product. For two 1-D arrays, `u @ v` is the **dot product**.
> * `np.linalg.det(A)`, `np.linalg.inv(A)`: the determinant and the inverse of a square matrix.
> * `np.linalg.solve(A, b)`: solves `A x = b` for `x`. More accurate and faster than `inv(A) @ b`.
> * `np.allclose(X, Y)`: compare results of floating-point calculations with a tolerance.

In [124]:
# 🟢 Element-wise arithmetic
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print(A + B)          # [[ 6  8] [10 12]]
print(A - B)          # [[-4 -4] [-4 -4]]
print(A * B)          # [[ 5 12] [21 32]]: element by element, NOT the matrix product
print(B / A)          # true division gives floats
print(A ** 2)         # each element squared, NOT A @ A
print(A * 10 + 1)     # a number applies to every element

[[ 6  8]
 [10 12]]
[[-4 -4]
 [-4 -4]]
[[ 5 12]
 [21 32]]
[[5.         3.        ]
 [2.33333333 2.        ]]
[[ 1  4]
 [ 9 16]]
[[11 21]
 [31 41]]


In [125]:
# 🟢 Matrix arithmetic
print(A @ B)                          # [[19 22] [43 50]]: row of A times column of B
print(1 * 5 + 2 * 7)                  # 19: the top-left element, by hand
print(A.T)                            # [[1 3] [2 4]]
print(np.array_equal(A @ np.eye(2), A))   # True: multiplying by the identity changes nothing

print(np.linalg.det(A))               # -2.0000000000000004: a float, so compare with a tolerance
A_inv = np.linalg.inv(A)
print(A_inv)                          # [[-2.   1. ] [ 1.5 -0.5]]
print(np.allclose(A @ A_inv, np.eye(2)))  # True: a matrix times its inverse is the identity

[[19 22]
 [43 50]]
19
[[1 3]
 [2 4]]
True
-2.0000000000000004
[[-2.   1. ]
 [ 1.5 -0.5]]
True


**What you should see:** `A * B` and `A @ B` give completely different results. In `A @ B`, the top-left element is row 0 of `A` times column 0 of `B`: 1 × 5 + 2 × 7 = 19. The determinant prints as `-2.0000000000000004` rather than −2: floating-point arithmetic (Week 1), so compare with `np.allclose`, never `==`.

### Shapes in matrix products, and vectors

In [126]:
# 🟢 (m, n) @ (n, p) -> (m, p): the inner sizes must match
X = np.ones((2, 3))
Y = np.ones((3, 4))
print((X @ Y).shape)                  # (2, 4)
try:
    X @ X                             # (2, 3) @ (2, 3): inner sizes 3 and 2 do not match
except ValueError as err:
    print("ValueError:", str(err)[:70])
print((X @ X.T).shape)                # (2, 2): transposing fixes it

u = np.array([1, 2, 3])
w = np.array([4, 5, 6])
print(u @ w)                          # 32: the dot product, 1*4 + 2*5 + 3*6
print(u * w)                          # [ 4 10 18]: element-wise, for comparison
print(u.reshape(3, 1) @ w.reshape(1, 3))   # the outer product: a 3 x 3 matrix

(2, 4)
ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with g
(2, 2)
32
[ 4 10 18]
[[ 4  5  6]
 [ 8 10 12]
 [12 15 18]]


**What you should see:** `(2, 3) @ (3, 4)` gives `(2, 4)`; `(2, 3) @ (2, 3)` fails because the **inner** sizes (3 and 2) differ; `X @ X.T` works. Two 1-D arrays give the dot product, 32. A column `(3, 1)` times a row `(1, 3)` gives the 3 × 3 outer product.

### Solving a system of equations

The system *2x + y = 5* and *x + 3y = 10* is `A x = b` with `A = [[2, 1], [1, 3]]` and `b = [5, 10]`.

In [127]:
# 🟢 Solve A x = b, then check the answer
A2 = np.array([[2.0, 1.0], [1.0, 3.0]])
b2 = np.array([5.0, 10.0])
x = np.linalg.solve(A2, b2)
print(x)                              # [1. 3.]: x = 1, y = 3
print(np.allclose(A2 @ x, b2))        # True: substituting back gives b

[1. 3.]
True


🔵 **Exercises.**

1. Predict, then compute, the shape of each product, storing the shapes in the list `shapes` (write `None` for a product that fails): `(3, 2) @ (2, 5)`, `(4, 4) @ (4,)`, `(2, 3) @ (2, 3)`.
2. With `P = np.array([[2, 0], [1, 3]])` and `Q = np.array([[1, 4], [2, 1]])`, store the element-wise product in `elementwise` and the matrix product in `matprod`. Check `matprod[1, 0]` by hand in a comment.
3. Is `P @ Q` equal to `Q @ P`? Store `True` or `False` in `commutes`. (Matrix multiplication is generally **not** commutative.)
4. Solve the system *x + y + z = 6*, *2y + 5z = −4*, *2x + 5y − z = 27*, storing the solution in `xyz`, and check it by substituting back.

In [128]:
P = np.array([[2, 0], [1, 3]])
Q = np.array([[1, 4], [2, 1]])

shapes = [(3, 5), (4,), None]          # (4, 4) @ (4,) gives a 1-D result; (2, 3) @ (2, 3) fails
elementwise = P * Q
matprod = P @ Q                        # matprod[1, 0] = 1*1 + 3*2 = 7
commutes = np.array_equal(P @ Q, Q @ P)
A3 = np.array([[1, 1, 1], [0, 2, 5], [2, 5, -1]], dtype=float)
b3 = np.array([6, -4, 27], dtype=float)
xyz = np.linalg.solve(A3, b3)
print(xyz, np.allclose(A3 @ xyz, b3))

[ 5.  3. -2.] True


In [129]:
# Tests for 2.3
check(shapes == [(3, 5), (4,), None], "Inner sizes must match; (n, n) @ (n,) gives shape (n,); (2, 3) @ (2, 3) fails.")
check(elementwise is not None and np.array_equal(elementwise, [[2, 0], [2, 3]]), "Element-wise: P * Q.")
check(matprod is not None and np.array_equal(matprod, [[2, 8], [7, 7]]), "Matrix product: P @ Q.")
check(commutes is False, "Compare P @ Q with Q @ P using np.array_equal.")
check(xyz is not None and np.allclose(xyz, [5, 3, -2]), "Build A and b from the three equations, then np.linalg.solve(A, b).")

✅ Correct
✅ Correct
✅ Correct
✅ Correct
✅ Correct


## 2.4 Putting it together: a small portfolio

A matrix of daily returns for three assets over five days, and portfolio weights. Matrix arithmetic answers portfolio questions in one line each: the **portfolio return** on each day is the returns matrix times the weight vector, `R @ w`, and the **portfolio variance** is `w @ C @ w`, where `C` is the covariance matrix of the returns.

In [130]:
# 🟢 Five days (rows) x three assets (columns), as decimals: 0.012 is +1.2%
R = np.array([[ 0.012, -0.004,  0.007],
              [-0.008,  0.010,  0.002],
              [ 0.005,  0.003, -0.006],
              [ 0.018, -0.002,  0.009],
              [-0.002,  0.006,  0.001]])
w = np.array([0.5, 0.3, 0.2])          # weights: they sum to 1
assets = np.array(["Equity", "Bonds", "Gold"])
print(R.shape, w.shape, w.sum())

(5, 3) (3,) 1.0


🔵 **Exercises.** Store each answer in the variable named.

1. `daily`: the portfolio's return on each day, a vector of 5 values (`R @ w`).
2. `best_day`: the **position** of the day with the highest portfolio return.
3. `mean_returns`: each asset's mean return over the five days (a reduction along the right axis).
4. `C`: the covariance matrix of the three assets, `np.cov(R, rowvar=False)` (each **column** is a variable).
5. `port_var`: the portfolio variance `w @ C @ w`, and `port_sd` its square root (a ufunc).
6. `above_mean`: a boolean matrix, True where an asset's return on a day is above **that asset's** mean (broadcasting a `(3,)` vector against a `(5, 3)` matrix).

In [131]:
daily = R @ w
best_day = daily.argmax()
mean_returns = R.mean(axis=0)
C = np.cov(R, rowvar=False)
port_var = w @ C @ w
port_sd = np.sqrt(port_var)
above_mean = R > mean_returns
print(daily.round(4), best_day, mean_returns.round(4), round(port_sd, 5))

[ 0.0062 -0.0006  0.0022  0.0102  0.001 ] 3 [0.005  0.0026 0.0026] 0.00437


In [132]:
# Tests for 2.4
check(daily is not None and np.allclose(daily, (R * w).sum(axis=1)), "R @ w: each day's returns times the weights, summed.")
check(best_day == 3, "argmax of the daily portfolio returns.")
check(mean_returns is not None and np.allclose(mean_returns, R.mean(axis=0)), "Average over the days: axis=0.")
check(C is not None and C.shape == (3, 3) and np.allclose(C, C.T), "np.cov(R, rowvar=False) gives a symmetric 3 x 3 matrix.")
check(port_var is not None and np.isclose(port_var, daily.var(ddof=1)), "w @ C @ w equals the sample variance of the daily portfolio returns.")
check(above_mean is not None and above_mean.shape == (5, 3), "R > mean_returns broadcasts the (3,) means across the 5 rows.")

✅ Correct
✅ Correct
✅ Correct
✅ Correct
✅ Correct
✅ Correct


**What you should see:** the portfolio's best day is day 3 (the fourth row), and the portfolio variance from `w @ C @ w` equals the variance of the daily portfolio returns computed directly: two routes to the same number, which is a good check that the matrix algebra is right.

✍️ **Reflect (two sentences).** Why does `R @ w` give one value per **day**, not per asset? What would `w @ R` do, and why does it fail?

*Your answer:*

---
# Part 3. NumPy: worked example, completion problem, unaided problem

The three problems share a pattern: a two-dimensional array of numbers, a **universal function** applied to all of it, **broadcasting** to combine it with a row or a column, and **indexing** to pick out the answer.

> **Method card: universal functions (ufuncs)** apply an operation element by element, in compiled code. Arithmetic operators are ufuncs (`a + b` calls `np.add`), and so are `np.sqrt`, `np.exp`, `np.log`, `np.abs`, `np.round`, `np.maximum` and `np.minimum`. Reductions such as `np.mean`, `np.std`, `np.argmax` take `axis=` and `keepdims=`.

> **Method card: `np.where(condition, x, y)`** returns `x` where the condition is True and `y` elsewhere, element by element: a vectorised `if`.

## 3.1 Worked example: standardising exam scores

Six students sat four exams with different difficulty. To compare across exams, we convert each score to a **z-score**: how many standard deviations it is from that exam's mean. Read each step and its output.

In [133]:
# 🟢 The data: 6 students (rows) x 4 exams (columns)
rng = np.random.default_rng(445)                          # a seeded Generator: same numbers every run
students = np.array(["Ana", "Ben", "Chen", "Dara", "Eli", "Femi"])
exams = np.array(["Stats", "Maths", "Econ", "History"])
scores = rng.normal(loc=[65, 55, 70, 60], scale=[10, 15, 8, 12], size=(6, 4)).round()
print(scores.shape)
print(scores)

(6, 4)
[[64. 59. 77. 77.]
 [76. 38. 69. 58.]
 [67. 42. 79. 74.]
 [70. 53. 72. 70.]
 [68. 59. 83. 73.]
 [75. 51. 69. 53.]]


`loc` and `scale` have shape `(4,)` and `size` is `(6, 4)`: NumPy **broadcasts** the four means and standard deviations across the six rows, so each column has its own mean and spread.

In [134]:
# 🟢 Step 1: each exam's mean and standard deviation, kept as shape (1, 4) for broadcasting
mean = scores.mean(axis=0, keepdims=True)                  # axis=0: the rows disappear
std = scores.std(axis=0, keepdims=True)
print("mean shape:", mean.shape, "| std shape:", std.shape)

# 🟢 Step 2: z-scores by broadcasting: (6, 4) minus (1, 4), divided by (1, 4)
z = (scores - mean) / std
print(z.round(2))
print("each exam now has mean", z.mean(axis=0).round(10), "and std", z.std(axis=0).round(10))

mean shape: (1, 4) | std shape: (1, 4)
[[-1.4   1.09  0.41  1.07]
 [ 1.4  -1.55 -1.11 -1.07]
 [-0.7  -1.05  0.79  0.73]
 [ 0.    0.34 -0.54  0.28]
 [-0.47  1.09  1.56  0.62]
 [ 1.17  0.08 -1.11 -1.64]]
each exam now has mean [ 0. -0.  0. -0.] and std [1. 1. 1. 1.]


**What you should see:** `mean` and `std` with shape `(1, 4)`, so subtracting them from the `(6, 4)` scores broadcasts down the rows: each score minus **its own exam's** mean. Each column of `z` now has mean 0 and standard deviation 1.

In [135]:
# 🟢 Step 3: indexing to answer questions
best = z.argmax(axis=0)                                    # row position of the best z-score in each exam
print("Top student per exam:", dict(zip(exams, students[best])))    # fancy indexing with an array of positions

strong = (z > 1).any(axis=1)                               # boolean mask: any exam more than 1 sd above the mean
print("At least one score above +1 sd:", students[strong])

overall = z.mean(axis=1)                                   # axis=1: the exams disappear, one value per student
ranking = np.argsort(overall)[::-1]                        # positions from best to worst
print("Overall ranking:", students[ranking])

# 🟢 Step 4: a ufunc and np.where: cap extreme z-scores at ±2, and label performance
capped = np.clip(z, -2, 2)
label = np.where(z >= 0, "above", "below")
print(label[:2])

Top student per exam: {np.str_('Stats'): np.str_('Ben'), np.str_('Maths'): np.str_('Ana'), np.str_('Econ'): np.str_('Eli'), np.str_('History'): np.str_('Ana')}
At least one score above +1 sd: ['Ana' 'Ben' 'Eli' 'Femi']
Overall ranking: ['Eli' 'Ana' 'Dara' 'Chen' 'Femi' 'Ben']
[['below' 'above' 'above' 'above']
 ['above' 'below' 'below' 'below']]


**What you should see:** a top student for each exam (`students[best]` uses **fancy indexing**: an array of positions picks several elements at once); the students with at least one strong exam (a **boolean mask** with `any(axis=1)`); an overall ranking from `argsort`; and a label array built by `np.where`.

Every operation worked on whole arrays: no Python loop over students or exams.

## 3.2 Completion problem: temperature anomalies

Monthly mean temperatures (°C) for three cities over six months. 🟡 Fill each `____`, using the worked example as a model.

In [136]:
# 🟢 The data
cities = np.array(["London", "Madrid", "Oslo"])
months = np.array(["Jan", "Feb", "Mar", "Apr", "May", "Jun"])
temps_c = np.array([[ 5.2,  5.5,  7.6, 10.0, 13.4, 16.5],
                    [ 6.3,  7.9, 11.2, 13.1, 17.0, 22.7],
                    [-2.9, -2.6,  0.6,  5.5, 10.8, 15.0]])
print(temps_c.shape)

(3, 6)


In [137]:
# 🟡 (a) Convert every value to Fahrenheit with one vectorised expression: F = C × 9/5 + 32
temps_f = temps_c * 9 / 5 + 32

# 🟡 (b) Each city's mean over the six months, kept as a column so it broadcasts back
city_mean = temps_c.mean(axis=1, keepdims=True)

# 🟡 (c) The anomaly: each month minus its own city's mean
anomaly = temps_c - city_mean

# 🟡 (d) The hottest month in each city: position with argmax along the months, then fancy indexing
hottest = months[temps_c.argmax(axis=1)]

# 🟡 (e) A boolean mask: months colder than 5 °C anywhere; then the number of such months per city
cold = temps_c < 5
cold_per_city = cold.sum(axis=1)

if filled(temps_f, city_mean, anomaly, hottest, cold_per_city):
    print(temps_f.round(1), "\n", city_mean.round(2), "\n", anomaly.round(2), "\n", hottest, cold_per_city)

[[41.4 41.9 45.7 50.  56.1 61.7]
 [43.3 46.2 52.2 55.6 62.6 72.9]
 [26.8 27.3 33.1 41.9 51.4 59. ]] 
 [[ 9.7 ]
 [13.03]
 [ 4.4 ]] 
 [[-4.5  -4.2  -2.1   0.3   3.7   6.8 ]
 [-6.73 -5.13 -1.83  0.07  3.97  9.67]
 [-7.3  -7.   -3.8   1.1   6.4  10.6 ]] 
 ['Jun' 'Jun' 'Jun'] [0 0 3]


In [138]:
# Tests for 3.2
check(np.isclose(temps_f[0, 0], 41.36), "F = C * 9 / 5 + 32.")
check(city_mean.shape == (3, 1), "Take the mean over the months (axis=1) with keepdims=True.")
check(np.allclose(anomaly.mean(axis=1), 0), "Subtract each city's own mean: anomalies should average 0 per city.")
check(list(hottest) == ["Jun", "Jun", "Jun"], "argmax over the months gives a position per city; index months with it.")
check(list(cold_per_city) == [0, 0, 3], "Sum the boolean mask over the months (True counts as 1).")

✅ Correct
✅ Correct
✅ Correct
✅ Correct
✅ Correct


✍️ **Reflect.** In (b), what goes wrong if you leave out `keepdims=True`? Try it, read the error, and explain it with the broadcasting rule in one sentence.

*Your answer:*

## 3.3 Unaided problem: distances between places

You have the coordinates of `n` points (here, positions on a map in kilometres). Write `pairwise_distances(points)` that returns an `(n, n)` array whose element `[i, j]` is the straight-line distance between point `i` and point `j`, **without any Python loop**. Then write `nearest(points)` that returns, for each point, the position of its nearest **other** point.

🔵 Hints, if you want them:

* Give the points two different extra axes so they broadcast against each other: `points[:, None, :]` has shape `(n, 1, 2)` and `points[None, :, :]` has shape `(1, n, 2)`. Their difference has shape `(n, n, 2)`.
* Square, sum over the last axis, and take the square root: three ufuncs.
* For `nearest`, a point is always at distance 0 from itself. Copy the distance matrix, set its diagonal to `np.inf` with `np.fill_diagonal`, then use `argmin` along one axis.

In [139]:
points = np.array([[0.0, 0.0], [2.0, 4.0], [6.0, 0.0], [2.5, -1.0]])

def pairwise_distances(points):
    diff = points[:, None, :] - points[None, :, :]          # (n, 1, 2) - (1, n, 2) -> (n, n, 2)
    return np.sqrt((diff ** 2).sum(axis=-1))                # (n, n)

def nearest(points):
    d = pairwise_distances(points).copy()
    np.fill_diagonal(d, np.inf)                             # a point is not its own neighbour
    return d.argmin(axis=1)

print(pairwise_distances(points).round(2))
print(nearest(points))

[[0.   4.47 6.   2.69]
 [4.47 0.   5.66 5.02]
 [6.   5.66 0.   3.64]
 [2.69 5.02 3.64 0.  ]]
[3 0 3 0]


In [140]:
# Tests for 3.3: compared with a slow, obviously correct loop
def slow(points):
    n = len(points)
    out = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            out[i, j] = ((points[i] - points[j]) ** 2).sum() ** 0.5
    return out

d = pairwise_distances(points)
check(d.shape == (4, 4), "The result should be (n, n).")
check(np.allclose(d, slow(points)), "Distances should match the loop version.")
check(np.allclose(d, d.T) and np.allclose(np.diag(d), 0), "The matrix should be symmetric with zeros on the diagonal.")
check(list(nearest(points)) == [3, 0, 3, 0], "Exclude each point itself before taking argmin.")
big = np.random.default_rng(1).random((300, 2))
check(pairwise_distances(big).shape == (300, 300), "It should work for any number of points.")

✅ Correct
✅ Correct
✅ Correct
✅ Correct
✅ Correct


✍️ **Reflect.** `diff` has shape `(n, n, 2)`. For 10,000 points, how many float64 values is that, and how much memory? What does that tell you about the cost of broadcasting?

*Your answer:*

---
# Part 4. Applied task: a properly typed DataFrame from your Week 2 JSON

In Week 2 you collected raw JSON from the **World Bank API**. Raw JSON loaded into pandas is rarely typed the way you want: numbers as text, repeated labels stored again and again, years as strings, empty strings instead of missing values. Your task is to turn it into a table where **every dtype is a deliberate choice**:

1. Load the raw records and measure **memory before**.
2. Flatten the JSON and inspect what pandas inferred.
3. Choose and apply a dtype for **every** column, with a reason.
4. Parse dates to **ISO 8601 in UTC**.
5. Set an **index for a reason**, and check it.
6. Measure **memory after**, and explain the difference.

## 4.1 Load the raw records

The loader looks for the raw World Bank files your Week 2 lab saved (`data/raw/worldbank_*.json` or the `cache/` pages). If you cloned your repository into Colab, or uploaded the files, it uses them. Otherwise it collects a fresh copy of the same kind of data: **population and GDP per capita for all countries, 2000 to 2023**, with the five professional steps from Week 2.

In [141]:
# 🟢 Optional: upload the raw World Bank JSON files you saved in Week 2
def upload_raw():
    from google.colab import files
    Path("data/raw").mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():
        (Path("data/raw") / name).write_bytes(data)
        print("Saved", name)

# upload_raw()          # remove the # to run

In [142]:
# 🟢 Load your Week 2 files, or collect a fresh copy
WB = "https://api.worldbank.org/v2"
HEADERS = {"User-Agent": "ST445 seminar lab (London School of Economics)"}

def records_from(body):
    """World Bank JSON comes as [metadata, records] pages, or as a saved list of records."""
    if isinstance(body, list) and len(body) == 2 and isinstance(body[0], dict) and "page" in body[0]:
        return body[0], body[1] or []
    return {}, body

def load_week2_files():
    files = sorted(Path("data/raw").glob("worldbank_*.json")) + sorted(Path("cache").glob("wb_all_*.json"))
    records, last_updated = [], set()
    for f in files:
        meta, recs = records_from(json.loads(f.read_text(encoding="utf-8")))
        records.extend(recs)
        if meta.get("lastupdated"):
            last_updated.add(meta["lastupdated"])
    return records, last_updated

def collect_fresh(indicators=("SP.POP.TOTL", "NY.GDP.PCAP.CD"), date="2000:2023"):
    records, last_updated = [], set()
    for ind in indicators:
        page = 1
        while True:
            r = requests.get(f"{WB}/country/all/indicator/{ind}",
                             params={"format": "json", "per_page": 1000, "date": date, "page": page},
                             headers=HEADERS, timeout=60)
            r.raise_for_status()                              # step 3: status before body
            meta, recs = records_from(r.json())               # step 4: the shape we expect
            records.extend(recs)
            last_updated.add(meta.get("lastupdated"))
            if page >= meta["pages"]:
                break
            page += 1
            time.sleep(0.5)
    return records, last_updated

raw_records, last_updated = load_week2_files()
source = "your Week 2 files"
if len(raw_records) < 500:                                    # too little to see memory effects: collect more
    raw_records, last_updated = collect_fresh()
    source = "a fresh World Bank collection"
print(f"{len(raw_records)} records from {source}; metadata lastupdated: {sorted(d for d in last_updated if d)}")
print(json.dumps(raw_records[0], indent=2))

12768 records from a fresh World Bank collection; metadata lastupdated: ['2026-10-08']
{
  "indicator": {
    "id": "SP.POP.TOTL",
    "value": "Population, total"
  },
  "country": {
    "id": "ZH",
    "value": "Africa Eastern and Southern"
  },
  "countryiso3code": "AFE",
  "date": "2023",
  "value": 750491370,
  "unit": "",
  "obs_status": "",
  "decimal": 0
}


**What you should see:** thousands of records, and the first one printed. Each record has nested `indicator` and `country` objects, a three-letter `countryiso3code`, a `date` (the year, **as text**), a `value`, and three fields that are usually empty: `unit`, `obs_status` and `decimal`.

## 4.2 Flatten, and measure memory before

> **Method card: `df.memory_usage(deep=True)`** returns the bytes used by each column. `deep=True` counts the text itself, not just the pointers to it; without it, text columns look far smaller than they are. `.sum()` gives the total.

In [143]:
# 🟢 Flatten exactly as received, then look at what pandas inferred
raw = pd.json_normalize(raw_records, sep="_")
print(raw.dtypes, "\n")
memory_before = raw.memory_usage(deep=True).sum()
print(f"Memory before: {memory_before / 1e6:.2f} MB for {len(raw)} rows")
raw.head()

countryiso3code     object
date                object
value              float64
unit                object
obs_status          object
decimal              int64
indicator_id        object
indicator_value     object
country_id          object
country_value       object
dtype: object 

Memory before: 5.93 MB for 12768 rows


,countryiso3code,date,value,unit,obs_status,decimal,indicator_id,indicator_value,country_id,country_value
0,AFE,2023,750491370.0,,,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern
1,AFE,2022,731821393.0,,,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern
2,AFE,2021,713090928.0,,,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern
3,AFE,2020,694446100.0,,,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern
4,AFE,2019,675950189.0,,,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern


**What you should see:** almost every column is text (`str` in pandas 3.0, `object` in 2.x); even `date`, which is a contains the year is also text. `value` is `float64` because missing values arrive as JSON `null`. `decimal` is an integer. Therefore, given that almost everything is text, we need to transform data types to ensure they are appropriate. To support if that, we can also look at how many distinct values each column has:

In [144]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12768 entries, 0 to 12767
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   countryiso3code  12768 non-null  object 
 1   date             12768 non-null  object 
 2   value            12554 non-null  float64
 3   unit             12768 non-null  object 
 4   obs_status       12768 non-null  object 
 5   decimal          12768 non-null  int64  
 6   indicator_id     12768 non-null  object 
 7   indicator_value  12768 non-null  object 
 8   country_id       12768 non-null  object 
 9   country_value    12768 non-null  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 997.6+ KB


We can also check how many unique values in each column. We can do this for each column separately as below:

In [145]:
raw.decimal.unique()

array([0, 1])

In [146]:
raw.indicator_id.unique()

array(['SP.POP.TOTL', 'NY.GDP.PCAP.CD'], dtype=object)

In [147]:
raw.date.unique()

array(['2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016',
       '2015', '2014', '2013', '2012', '2011', '2010', '2009', '2008',
       '2007', '2006', '2005', '2004', '2003', '2002', '2001', '2000'],
      dtype=object)

Alternatively, we can also sum how many unique values exist. For instance, check the code below that calculates unique values in all columns at once.

In [148]:
# 🟢 How many distinct values does each column have? Empty strings count as a value here.
pd.DataFrame({"dtype": raw.dtypes.astype(str), "distinct": raw.nunique(), "empty_strings": (raw == "").sum()})

,dtype,distinct,empty_strings
countryiso3code,object,262,240
date,object,24,0
value,float64,12459,0
unit,object,1,12768
obs_status,object,1,12768
decimal,int64,2,0
indicator_id,object,2,0
indicator_value,object,2,0
country_id,object,266,0
country_value,object,266,0


**What you should see:** `indicator_id` and `indicator_value` have only 2 distinct values acrowss the thousands of rows (or a few, if your Week 2 files used other indicators); `country_value` and `countryiso3code` a few hundred; `unit` and `obs_status` are mostly or entirely empty strings, which should be consider as missing values.

When to use string data type (`str` or `object`) and `category`? Conventionally, repeated labels are what `category` is for. There are some differences also in memory usage and processing speed, as shown in the table below.


| Feature | `str` / `object` | `category` |
|----------|------------------|------------|
| Best used for | Unique or highly varied text<br>(names, IDs, notes, comments) | Repeated labels or classes<br>(e.g., gender, status, country) |
| Memory footprint | Stores every string value individually.<br>If `"apple"` appears 10,000 times,<br>it is stored 10,000 times. | Stores unique values once.<br>Rows store small integer codes<br>pointing to those values. |
| Execution speed | Compares full text strings.<br>Groupbys, merges, and filters<br>are generally slower. | Compares integer codes.<br>Groupbys, merges, and filters<br>are generally faster. |

## 4.3 Choose a dtype for every column, with a reason

✍️ Complete the decision table **before** you write the code. One line per column.

| Column | Inferred dtype | Your dtype | Your reason |
|---|---|---|---|
| `indicator_id` | *object* |*category* |2 values only, repeated thousands of times. Therefore, category is better as it uses less memory |
| `indicator_value` |*object* |*category* | 2 values only, repeated thousands of times. Uses less memory |
| `country_id` | *float64*|*category* | A few hundred of ISO code values, repeated every year |
| `country_value` |*object* |*category* | Same reason as above |
| `countryiso3code` | *object* |*category* | Same reason as above |
| `date` |*int64* | unchanged | |
| `value` |*object* | *float64*| Because this is a decimal number |
| `unit` | *object*| unchanged | |
| `obs_status` |*object* | unchanged| |
| `decimal` |*object* |*int8*  | This is an integer, and there are only two numbers in the column |

> **Method card: choosing dtypes in pandas**
>
> | Use | When | How |
> |---|---|---|
> | `category` | a small set of labels repeated many times | `.astype("category")` |
> | `"string"` | free text, or codes you will not group by | `.astype("string")`: missing values become `pd.NA`, in pandas 2.x and 3.0 alike |
> | `Int16`, `Int64` (nullable) | whole numbers that may be missing | `.astype("Int16")`; the capital I keeps integers with `pd.NA` |
> | `float64` | measurements with decimals or missing values | usually already right |
> | drop the column | it holds no information | `df.drop(columns=[...])`, and say why |
>
> `df.replace("", pd.NA)` turns empty strings into proper (pandas recognisable) missing values first.

In [149]:
typed = raw.replace("", pd.NA)                         # copy / empty strings are recognised as missing values (NA)

In [150]:
typed.head()

,countryiso3code,date,value,unit,obs_status,decimal,indicator_id,indicator_value,country_id,country_value
0,AFE,2023,750491370.0,<NA>,<NA>,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern
1,AFE,2022,731821393.0,<NA>,<NA>,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern
2,AFE,2021,713090928.0,<NA>,<NA>,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern
3,AFE,2020,694446100.0,<NA>,<NA>,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern
4,AFE,2019,675950189.0,<NA>,<NA>,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern


Now that we appropriately transformed missing values to a recognisable pandas type, we can calculate how many exist.

In [151]:
typed.isna().sum()

,0
countryiso3code,240
date,0
value,214
unit,12768
obs_status,12768
decimal,0
indicator_id,0
indicator_value,0
country_id,0
country_value,0


Observe that it matches the total number of empty strings we calculated before. Now, lets transform the dtype of each column as appropriate.

In [152]:
# 🟡 Apply your decisions. Replace each ____ with a dtype (or fill the list of columns to drop).
DTYPES = {
    "indicator_id":    "category",   # given there are only 2 values repeated thousands of times, category is better to reduce memory
    "indicator_value": "category",   # the indicator's name: same reason as above.
    "country_id":      "category",   # ISO2 codes: a few hundred values only, repeated every year. So, category is also better.
    "country_value":   "category",   # There are only a few hundred country name. So, also chose category to reduce memory.
    "countryiso3code": "category",   # the join key: repeated every year and indicator
    "value":           "float64",    # a measurement with decimals and missing values
    "decimal":         "Int8",       # 0 to 2: a tiny whole number that needs no more than 8 bits
}
DROP = [c for c in ["unit", "obs_status"] if typed[c].isna().all()]   # checks for entirely empty column: no information. So, better to just drop them.

if filled(DTYPES, DROP):
    typed = typed.drop(columns=DROP).astype(DTYPES) # drop all missing values from "unit" and "obs_status", otherwise transform the dtypes according to DTYPES
    typed["year"] = typed.pop("date").astype("Int16")  # the year as a small nullable integer
    print(typed.dtypes)

countryiso3code    category
value               float64
decimal                Int8
indicator_id       category
indicator_value    category
country_id         category
country_value      category
year                  Int16
dtype: object


Now that we removed (dropped 'unit', 'obs_status']) we can see in the table that they are not available anymore.

In [153]:
typed.head()

,countryiso3code,value,decimal,indicator_id,indicator_value,country_id,country_value,year
0,AFE,750491370.0,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern,2023
1,AFE,731821393.0,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern,2022
2,AFE,713090928.0,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern,2021
3,AFE,694446100.0,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern,2020
4,AFE,675950189.0,0,SP.POP.TOTL,"Population, total",ZH,Africa Eastern and Southern,2019


**Something to decide:** if `unit` or `obs_status` is not entirely empty in your data, do not drop it: make it `category` and say why in your table. The test below accepts either, as long as no column is left as general-purpose text.

Now, let's check if there are any values still missing:

In [154]:
typed.isna().sum()

,0
countryiso3code,240
value,214
decimal,0
indicator_id,0
indicator_value,0
country_id,0
country_value,0
year,0


Therefore, we still have missing values in ['countryiso3code',['value']). We need to either remove or replace. In this case, we will drop given the number is not significant. However, you might want to replace for an identifier that can clarify that in your project.

In [155]:
# Drop rows where 'countryiso3code' is missing (NA)
typed = typed.dropna(subset=['countryiso3code'])
typed = typed.dropna(subset=['value'])
typed.isna().sum()

,0
countryiso3code,0
value,0
decimal,0
indicator_id,0
indicator_value,0
country_id,0
country_value,0
year,0


Now, we have no more missing values.

In [156]:
# Test for 4.3
left_as_text = [c for c in typed.columns if typed[c].dtype == object or str(typed[c].dtype) == "str"]
check(not left_as_text, f"These columns are still general-purpose text: {left_as_text}")
check(str(typed["countryiso3code"].dtype) in ("category", "string"), "The ISO3 code should be category (or string).")
check(str(typed["year"].dtype) == "Int16", "The year should be a nullable integer.")

✅ Correct
✅ Correct
✅ Correct


## 4.4 Dates in ISO 8601, in UTC

Two kinds of date are in this data: the **period** each value refers to (a year), and the date the World Bank **last updated** the series (in the metadata). Store both as time-zone-aware datetimes in UTC, then check they print as ISO 8601.

> **Method card: `pd.to_datetime(values, format=..., utc=True)`**
>
> * `format`: the exact pattern of the input, such as `"%Y"` for a year or `"%Y-%m-%d"` for a date. Give it whenever you know it, so nothing is guessed.
> * `utc=True`: returns times in UTC, the zone to store in. Convert to local time only for display.
>
> `series.dt.strftime("%Y-%m-%dT%H:%M:%SZ")` writes the ISO 8601 text form, with `Z` for UTC.

In [157]:
# 🟡 The input patterns: a four-digit year, and a date such as "2026-07-01"
YEAR_FORMAT = "%Y"
DATE_FORMAT = "%Y-%m-%d"

if filled(YEAR_FORMAT, DATE_FORMAT):
    typed["period_start"] = pd.to_datetime(typed["year"].astype("string"), format=YEAR_FORMAT, utc=True)
    updated = sorted(d for d in last_updated if d)
    typed["source_last_updated"] = pd.to_datetime(updated[-1] if updated else None, format=DATE_FORMAT, utc=True)
    print(typed[["year", "period_start", "source_last_updated"]].head(3))
    print(typed["period_start"].dt.strftime("%Y-%m-%dT%H:%M:%SZ").iloc[0])

   year              period_start       source_last_updated
0  2023 2023-01-01 00:00:00+00:00 2026-10-08 00:00:00+00:00
1  2022 2022-01-01 00:00:00+00:00 2026-10-08 00:00:00+00:00
2  2021 2021-01-01 00:00:00+00:00 2026-10-08 00:00:00+00:00
2023-01-01T00:00:00Z


In [158]:
# Test for 4.4
check(str(typed["period_start"].dt.tz) == "UTC", "Use utc=True.")
check(typed["period_start"].dt.strftime("%Y-%m-%dT%H:%M:%SZ").str.fullmatch(r"\d{4}-01-01T00:00:00Z").all(),
      "Each year should become YYYY-01-01T00:00:00Z.")

✅ Correct
✅ Correct


**What you should see:** `period_start` such as `2023-01-01 00:00:00+00:00`, which prints in ISO 8601 as `2023-01-01T00:00:00Z`. If your Week 2 files had no metadata, `source_last_updated` is `NaT`: the datetime version of a missing value.

## 4.5 An index, set for a reason

First, we must understand what **one row** represents. As we have seen above, eahch row contains the value of **one indicator, for one country, in one year**. Given this, the natural key in the table would be `(countryiso3code, indicator_id, year)`: it identifies each row, and it is the key you will join on (Week 2) and select by.

🟡 Set the index, check it is unique, and sort it so label slices work.

In [159]:
KEY = ["countryiso3code", "indicator_id", "year"]

if filled(KEY):
    table = typed.set_index(KEY).sort_index()
    print("unique:", table.index.is_unique, "| levels:", table.index.names)
    print(table.head())

unique: True | levels: ['countryiso3code', 'indicator_id', 'year']
                                            value  decimal  \
countryiso3code indicator_id   year                          
ABW             NY.GDP.PCAP.CD 2000  20681.023027        1   
                               2001  20740.132583        1   
                               2002  21307.248251        1   
                               2003  21949.485996        1   
                               2004  23700.631990        1   

                                                  indicator_value country_id  \
countryiso3code indicator_id   year                                            
ABW             NY.GDP.PCAP.CD 2000  GDP per capita (current US$)         AW   
                               2001  GDP per capita (current US$)         AW   
                               2002  GDP per capita (current US$)         AW   
                               2003  GDP per capita (current US$)         AW   
                  

In [160]:
# Test for 4.5
check(table.index.is_unique, "The key should identify each row exactly once. Did two Week 2 files overlap? Try typed.drop_duplicates(KEY).")
check(table.index.is_monotonic_increasing, "Sort the index with sort_index().")

✅ Correct
✅ Correct


✍️ **Your reason (two sentences).** Why this key and not just the country code? What would `table.index.is_unique` being `False` tell you about your Week 2 collection?

*Your answer:* If `table.index.is_unique` == `False` that mean we would have keys that repeat. This would have happened, for example, if we haven't dropped missing values from `['countryiso3code','value']`. Missing values in country would be considered duplicates causing indexes not to be unique. However, given we decided to drop the missing values, they are unique.

*try this code again without dropping missing rows in `['countryiso3code','value']` and see what's changed.*



## 4.6 Memory after, and what changed

In [161]:
# 🟢 Memory after, column by column
memory_after = table.memory_usage(deep=True, index=True).sum()
print(f"Memory before: {memory_before / 1e6:.2f} MB")
print(f"Memory after:  {memory_after / 1e6:.2f} MB  ({memory_after / memory_before:.0%} of before)\n")

compare = pd.DataFrame({"before_bytes": raw.memory_usage(deep=True)})
compare["after_bytes"] = table.reset_index().memory_usage(deep=True)
compare.sort_values("before_bytes", ascending=False)

Memory before: 5.93 MB
Memory after:  0.77 MB  (13% of before)



,before_bytes,after_bytes
indicator_value,912912,12613.0
country_value,787872,49434.0
indicator_id,785232,12593.0
date,676704,NaN
countryiso3code,663216,46592.0
country_id,651168,46586.0
obs_status,625632,NaN
unit,625632,NaN
decimal,102144,24724.0
value,102144,98896.0


**What you should see:** a much smaller total, typically a fraction of the original. The big savings come from the repeated labels (`indicator_value`, `country_value`) becoming `category`: each distinct name is stored once, plus a small integer code per row. The two new datetime columns add a little. (Exact figures depend on your pandas version and whether PyArrow is installed.)

✍️ **Report (three sentences).** State memory before and after. Which two columns saved most, and why? Name one dtype decision that was not about memory at all.

*Your answer:* `indicator_value`, `country_value`, and `indicator_id` were the features (columns) with the smallest number of unique values. Therefore, transforming them into category considerably reduced memory usage. Simiarly to `countryiso3code` and `country_id`. Therefore, converting them to `category` reduces memory significantly. Limiting decimal to `Int8` rather `int64` also reduces memory significantly as it uses less bits to represent data.

In [162]:
# 🟢 Save the typed table. Parquet keeps the dtypes; CSV would turn them back into text (Week 1).
Path("data/processed").mkdir(parents=True, exist_ok=True)
table.to_parquet("data/processed/worldbank_typed.parquet")
back = pd.read_parquet("data/processed/worldbank_typed.parquet")
check(back.dtypes.equals(table.dtypes), "The dtypes should survive the round trip through Parquet.")

✅ Correct


---
# Part 5. Error hunt

Each cell below contains a bug from the lecture's "Be careful" slides. For each: run it, ✍️ say in one sentence what is wrong, then 🟡 fix it in the cell underneath so the test passes.

## 5.1 Centring each row

In [163]:
# 🟢 Broken: subtract each row's mean
x = np.arange(12.0).reshape(4, 3)
row_mean = x.mean(axis=1)
try:
    centred = x - row_mean
except ValueError as err:
    print("ValueError:", err)

ValueError: operands could not be broadcast together with shapes (4,3) (4,) 


✍️ *What is wrong:*

In [164]:
# 🟡 Fix
centred = x - x.mean(axis=1, keepdims=True)

In [165]:
check(centred.shape == (4, 3) and np.allclose(centred.mean(axis=1), 0), "Each row should have mean 0. Keep the axis with keepdims=True, or use row_mean[:, None].")

✅ Correct


## 5.2 A price list with a tax rate per item

The intention: each of 3 prices multiplied by its own tax factor, giving 3 values.

In [166]:
# 🟢 Broken: it runs, but is it right?
prices = np.array([[10.0], [20.0], [30.0]])     # a column, shape (3, 1)
tax = np.array([1.2, 1.05, 1.0])                # shape (3,)
with_tax = prices * tax
print(with_tax.shape)
print(with_tax)

(3, 3)
[[12.  10.5 10. ]
 [24.  21.  20. ]
 [36.  31.5 30. ]]


✍️ *What is wrong (look at the shape):*

In [167]:
# 🟡 Fix: three prices, each times its own tax factor
with_tax = prices.ravel() * tax

In [168]:
check(np.shape(with_tax) == (3,) and np.allclose(with_tax, [12.0, 21.0, 30.0]), "Flatten the column first (ravel), so both have shape (3,).")

✅ Correct


## 5.3 Flagging high values in a DataFrame

In [169]:
# 🟢 Broken: flag countries with GDP per capita above 40,000
gdp = pd.DataFrame({"iso3": ["GBR", "IND", "USA"], "gdp_pc": [48_000, 2_400, 76_000]})
gdp["rich"] = False
gdp[gdp["gdp_pc"] > 40_000]["rich"] = True       # chained assignment
print(gdp)

  iso3  gdp_pc   rich
0  GBR   48000  False
1  IND    2400  False
2  USA   76000  False


/tmp/ipykernel_1140/3475758717.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gdp[gdp["gdp_pc"] > 40_000]["rich"] = True       # chained assignment


✍️ *What is wrong:* (In pandas 3.0 you also see a `ChainedAssignmentError` warning; in 2.x, a `SettingWithCopyWarning`. Either way, `gdp` did not change.)

In [170]:
# 🟡 Fix with a single .loc
gdp["rich"] = False
gdp.loc[gdp["gdp_pc"] > 40_000, "rich"] = True

In [171]:
check(gdp["rich"].tolist() == [True, False, True], "Use gdp.loc[condition, 'rich'] = True: rows and column in one selection.")

✅ Correct


## 5.4 A second kind of chained assignment

This one is more dangerous, because whether it works **depends on the pandas version**: in pandas 2.x it usually changes `gdp` (with a warning); in pandas 3.0 it never does.

In [172]:
# 🟢 Broken: cap GDP per capita at 50,000
gdp["capped"] = gdp["gdp_pc"]
gdp["capped"][gdp["capped"] > 50_000] = 50_000   # column first, then rows: chained
print(gdp)

  iso3  gdp_pc   rich  capped
0  GBR   48000   True   48000
1  IND    2400  False    2400
2  USA   76000   True   50000


/tmp/ipykernel_1140/3317418592.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  gdp["capped"][gdp["capped"] > 50_000] = 50_000   # column first, then rows: chained
/tmp/ipykernel_1140/3317418592.py:3: SettingWithCopyWarning: 
A value is t

✍️ *What is wrong, and why is code whose behaviour depends on the version a problem for reproducibility (Week 1)?*

In [173]:
# 🟡 Fix: one way with .loc, and one way with a ufunc that needs no assignment at all
gdp["capped"] = gdp["gdp_pc"]
gdp.loc[gdp["capped"] > 50_000, "capped"] = 50_000   # with .loc

gdp["capped_ufunc"] = np.minimum(gdp["gdp_pc"], 50_000)   # with np.minimum

In [174]:
check(gdp["capped"].tolist() == [48_000, 2_400, 50_000], "Use gdp.loc[mask, 'capped'] = 50_000.")
check(gdp["capped_ufunc"].tolist() == [48_000, 2_400, 50_000], "np.minimum compares element by element with a scalar (broadcasting).")

✅ Correct
✅ Correct


---
# Part 6. Extension: vectorise a simulation

A **random walk** starts at 0 and, at each of `n_steps` steps, moves +1 or −1 with equal probability. Simulating many walks answers questions such as: *what fraction of walks ever reach +10 within 100 steps?*

Below is a loop version. 🔵 Write a vectorised version that uses **one call** to the Generator for all the steps of all the walks, `cumsum` along the steps, and a reduction along the steps. Then compare the results and the runtimes.

In [175]:
# 🟢 The loop version (slow, but obviously correct)
def walks_loop(n_walks, n_steps, target, seed=445):
    rng = np.random.default_rng(seed)
    hits = 0
    finals = []
    for _ in range(n_walks):
        position, reached = 0, False
        for _ in range(n_steps):
            position += 1 if rng.random() < 0.5 else -1
            if position >= target:
                reached = True
        hits += reached
        finals.append(position)
    return hits / n_walks, np.array(finals)

share, finals = walks_loop(2_000, 100, 10)
print(f"Share reaching +10: {share:.3f} | mean final position {finals.mean():.2f} | variance {finals.var():.1f}")

Share reaching +10: 0.295 | mean final position -0.53 | variance 97.5


🔵 Write `walks_vectorised(n_walks, n_steps, target, seed=445)` returning the same two things. Hints:

* `rng.choice([-1, 1], size=(n_walks, n_steps))`: one row per walk, one column per step.
* `.cumsum(axis=1)`: the position after each step.
* `(positions >= target).any(axis=1)`: did each walk reach the target?
* The final position is the last column.

The two versions draw random numbers in a different order, so their results will not be identical, only close. Theory says the final position has mean 0 and variance `n_steps`.

In [176]:
def walks_vectorised(n_walks, n_steps, target, seed=445):
    rng = np.random.default_rng(seed)
    steps = rng.choice([-1, 1], size=(n_walks, n_steps))
    positions = steps.cumsum(axis=1)
    share = (positions >= target).any(axis=1).mean()
    return share, positions[:, -1]

share_v, finals_v = walks_vectorised(2_000, 100, 10)
print(f"Share reaching +10: {share_v:.3f} | mean final position {finals_v.mean():.2f} | variance {finals_v.var():.1f}")

Share reaching +10: 0.321 | mean final position 0.32 | variance 100.5


In [177]:
# Tests for Part 6
share_v, finals_v = walks_vectorised(20_000, 100, 10)
check(finals_v.shape == (20_000,), "Return one final position per walk.")
check(abs(finals_v.mean()) < 0.3 and abs(finals_v.var() - 100) < 5, "Final positions should have mean about 0 and variance about n_steps.")
check(abs(share_v - walks_loop(2_000, 100, 10)[0]) < 0.06, "The share should be close to the loop version's.")

✅ Correct
✅ Correct
✅ Correct


In [178]:
# 🟢 Compare runtimes: time.perf_counter measures wall-clock time
for n in [100, 1_000, 5_000]:
    t0 = time.perf_counter(); walks_loop(n, 100, 10); t_loop = time.perf_counter() - t0
    t0 = time.perf_counter(); walks_vectorised(n, 100, 10); t_vec = time.perf_counter() - t0
    print(f"{n:>6} walks: loop {t_loop * 1000:8.1f} ms | vectorised {t_vec * 1000:6.2f} ms | {t_loop / t_vec:6.0f}x faster")

   100 walks: loop      6.4 ms | vectorised   1.93 ms |      3x faster
  1000 walks: loop     78.6 ms | vectorised   1.39 ms |     57x faster
  5000 walks: loop    294.0 ms | vectorised   8.86 ms |     33x faster


**What you should see:** the vectorised version is many times faster, and the gap grows with the number of walks.

✍️ **Reflect (three sentences).** For 1,000,000 walks of 1,000 steps, how much memory would the `(n_walks, n_steps)` array of steps need as `int64`? Which dtype could you use instead, and what would processing the walks in chunks gain? (Lecture: vectorised code trades memory for speed.)

For a more careful timing, use `%timeit walks_vectorised(5_000, 100, 10)` in its own cell: it runs the code several times and reports the spread.

*Your answer:*

---
## Before you leave

- [ ] Part 1: you wrote a prediction before running each cell.
- [ ] Parts 2 to 5: every test shows ✅.
- [ ] Part 4: your decision table, index reason and memory report are written.
- [ ] **Edit → Clear all outputs**, download the notebook into your personal repository and commit it. Commit `data/processed/worldbank_typed.parquet` only if your `.gitignore` allows processed data; otherwise keep it local.